# Notebook 1: Can Weights Emerge from Practice?

Notebook 0 showed the aggregation model works **when handed good weights**. This
notebook asks the first practice-grounding question, which is the real test of
Berman's theory: can the relative importance of principles emerge from judges
deciding cases, with no philosopher setting it?

Berman's answer is that principles are "constituted by the taking-up behaviors of
the system's subscribers" (JESP p. 302), and that a norm becomes a genuine legal
principle only once enough officials take it up; a single invocation does not
make a principle of the system (p. 321). We model that directly. A bench of
judges, each carrying an interpretive methodology that sets how often it invokes
each principle, decides a stream of cases. A principle's emergent **uptake share**
is how often the bench takes it up.

**What this notebook claims.** A stable separation emerges: some principles are
taken up far more than others, the membership of the high and low groups is the
same across runs, and applying Berman's uptake threshold turns the low group into
a "dead" (sub-legal) cluster. This split is the legal/sub-legal boundary Berman's
account requires, and it survives whatever rule we use to replace judges over
time, so it is a property of the bench's heterogeneity, not of any optimizer. We
then check that the emergent weights still satisfy the requirements Notebook 0
established (R1-R4).

**What this notebook does not claim.** The emergent quantity is invocation
frequency off a stipulated set of methodologies. It is *uptake share*, not
Berman's full notion of weight, which is conferred by a principle winning
conflicts ("by battle," p. 324) and by being relied upon in later decisions
(p. 322). Neither is modeled here; a principle invoked constantly but that always
loses would have high uptake share and low Berman weight. Those mechanisms are
the subject of Notebook 1c. So this notebook shows a defensible legal/sub-legal
boundary emerges; it stops short of "practice grounds the weights."

## Setup
As in Notebook 0, the first cell writes and imports the shared core, which now
also contains the agent-based model (`LegalSystem`, the four methodologies, the
four turnover rules, and the bimodality tests).

In [1]:
import os
PROJECT = '/content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Run_2026-06'
try:
    from google.colab import drive            # on Colab: mount Drive and work in the project folder
    drive.mount('/content/drive')
    os.makedirs(PROJECT, exist_ok=True); os.chdir(PROJECT)
    print('Colab: mounted Drive, working in', os.getcwd())
except ImportError:
    print('Local: working in', os.getcwd())

Mounted at /content/drive
Colab: mounted Drive, working in /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Run_2026-06


In [2]:
%%writefile berman_core.py
"""
berman_core.py — shared, corrected core for the Berman Modeling rebuild (2026-06).

ONE source of truth for the SAW model, principle sets, Hart baselines, a *seeded*
bootstrap, the activation generators, the weight-tolerance (critical-sigma)
machinery, and the redesigned evaluation criteria. Every notebook %%writefile's
this file and `from berman_core import *`, so a fix here propagates everywhere
(ends the verbatim-copy drift of the old notebooks).

Design decisions baked in (vs. the old code):
  * Determinacy convention is strict |F| > tau, matching the manuscript.
  * RNG is always explicit (np.random.default_rng(seed)); bootstrap is seeded
    and vectorized, so every CI is reproducible.
  * One determinacy cutoff (DEFAULT_TAU) everywhere — no 0.05-here-0.2-there.
  * Corrected category map: federalism is STRUCTURAL, not moral.
  * Criteria are split into REQUIREMENTS (R1-R4, pass/fail, derived from what
    Berman actually claims) and DESCRIPTIVE measures (D1-D2, reported, never used
    to fail the theory). See CONCEPTUAL_AUDIT_vs_Berman.md.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Callable, Tuple

import numpy as np

__all__ = [
    "DEFAULT_TAU", "Principle", "NAMED_PRINCIPLES", "CATEGORY_INDEX",
    "make_weights", "named_weight_vector", "category_of",
    "net_force", "is_determinate", "determinacy_rate",
    "random_activations", "category_correlated_activations", "fully_correlated_activations",
    "bootstrap_ci", "critical_sigma", "CRITICAL_SIGMA_CEIL",
    "hart_hierarchical_determinate", "hart_consensus_determinate", "hart_advantage",
    "hart_validation_status",
    "nonlexicality_score", "ordinalize", "saw_verdict", "lexicographic_verdict",
    "weight_cv", "n_survivors", "cluster_gap",
    "METHODOLOGIES", "methodology_tendencies", "LegalSystem",
    "gmm_bimodal", "bimodality_coefficient",
    "RelianceSystem", "weight_entropy", "gini", "n_bins_occupied",
]

# ---------------------------------------------------------------------------
# Determinacy convention
# ---------------------------------------------------------------------------
DEFAULT_TAU: float = 0.20  # a case is determinate when |F| > tau (strict)


# ---------------------------------------------------------------------------
# Principles and categories  (federalism corrected to 'structural')
# ---------------------------------------------------------------------------
@dataclass
class Principle:
    name: str
    weight: float
    category: str = "structural"  # one of: 'textual', 'structural', 'moral'

    def __post_init__(self) -> None:
        assert 0.0 <= self.weight <= 1.0, f"weight out of [0,1]: {self.weight}"
        assert self.category in ("textual", "structural", "moral"), self.category


# The baseline 9-principle set. Order is fixed; categories are the corrected map.
NAMED_PRINCIPLES: List[Tuple[str, str]] = [
    ("text",       "textual"),
    ("original",   "textual"),
    ("precedent",  "structural"),
    ("structure",  "structural"),
    ("federalism", "structural"),   # <-- was grouped with 'moral' in the old code (bug)
    ("dignity",    "moral"),
    ("equality",   "moral"),
    ("liberty",    "moral"),
    ("welfare",    "moral"),
]

CATEGORY_INDEX: Dict[str, List[int]] = {
    cat: [i for i, (_, c) in enumerate(NAMED_PRINCIPLES) if c == cat]
    for cat in ("textual", "structural", "moral")
}


def category_of(i: int) -> str:
    return NAMED_PRINCIPLES[i][1]


# ---------------------------------------------------------------------------
# Weight generators
# ---------------------------------------------------------------------------
def make_weights(n: int, distribution: str, seed: int) -> np.ndarray:
    """Generate n weights. 'uniform' ~U(.60,.70); 'moderate' ~U(.50,.80);
    'high' ~U(.30,.95). Berman's weights are 'rough' (slight..nearly conclusive)."""
    rng = np.random.default_rng(seed)
    if distribution == "uniform":
        return rng.uniform(0.60, 0.70, n)
    if distribution == "moderate":
        return rng.uniform(0.50, 0.80, n)
    if distribution == "high":
        return rng.uniform(0.30, 0.95, n)
    raise ValueError(f"unknown distribution: {distribution}")


def named_weight_vector(distribution: str = "moderate", seed: int = 0) -> np.ndarray:
    """Weights for the 9 NAMED_PRINCIPLES under a given distribution."""
    return make_weights(len(NAMED_PRINCIPLES), distribution, seed)


# ---------------------------------------------------------------------------
# SAW model (nonlexical aggregation)
# ---------------------------------------------------------------------------
def net_force(weights: np.ndarray, activations: np.ndarray) -> np.ndarray:
    """F = sum_i w_i * a_i. Vectorized over a (..., n) activation array."""
    weights = np.asarray(weights, float)
    activations = np.asarray(activations, float)
    return np.sum(weights * activations, axis=-1)


def is_determinate(weights, activations, tau: float = DEFAULT_TAU):
    return np.abs(net_force(weights, activations)) > tau


def determinacy_rate(weights, activation_matrix, tau: float = DEFAULT_TAU) -> float:
    return float(np.mean(is_determinate(weights, activation_matrix, tau)))


# ---------------------------------------------------------------------------
# Activation generators (the conservative independent baseline + realistic structure)
# ---------------------------------------------------------------------------
def random_activations(n_cases: int, n_principles: int, rng: np.random.Generator) -> np.ndarray:
    """Independent activations ~ U(-1, 1). The conservative baseline."""
    return rng.uniform(-1.0, 1.0, size=(n_cases, n_principles))


def category_correlated_activations(n_cases: int, rng: np.random.Generator,
                                    rho: float = 0.7) -> np.ndarray:
    """Activations for the 9 NAMED_PRINCIPLES where principles in the same
    category share a common latent factor (textual together, structural together,
    moral together)."""
    n_p = len(NAMED_PRINCIPLES)
    a = np.empty((n_cases, n_p))
    for cat, idx in CATEGORY_INDEX.items():
        factor = rng.uniform(-1.0, 1.0, size=(n_cases, 1))
        idio = rng.uniform(-1.0, 1.0, size=(n_cases, len(idx)))
        a[:, idx] = np.clip(rho * factor + (1 - rho) * idio, -1.0, 1.0)
    return a


def fully_correlated_activations(n_cases: int, n_principles: int,
                                 rng: np.random.Generator, rho: float = 0.7) -> np.ndarray:
    """All principles share one latent factor."""
    factor = rng.uniform(-1.0, 1.0, size=(n_cases, 1))
    idio = rng.uniform(-1.0, 1.0, size=(n_cases, n_principles))
    return np.clip(rho * factor + (1 - rho) * idio, -1.0, 1.0)


# ---------------------------------------------------------------------------
# Bootstrap — SEEDED and vectorized (every CI is reproducible)
# ---------------------------------------------------------------------------
def bootstrap_ci(values, n_boot: int = 2000, ci: int = 95, seed: int = 0):
    """Return (mean, lo, hi) for a percentile bootstrap CI on the mean."""
    rng = np.random.default_rng(seed)
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return (float("nan"),) * 3
    means = rng.choice(arr, size=(n_boot, arr.size), replace=True).mean(axis=1)
    lo = float(np.percentile(means, (100 - ci) / 2))
    hi = float(np.percentile(means, 100 - (100 - ci) / 2))
    return float(arr.mean()), lo, hi


# ---------------------------------------------------------------------------
# Weight-uncertainty tolerance: critical sigma  (R2 — "rough weights suffice")
# ---------------------------------------------------------------------------
CRITICAL_SIGMA_CEIL = 1.0  # grid maximum; a returned value == this is right-censored


def critical_sigma(weights: np.ndarray, activation_matrix: np.ndarray,
                   tau: float = DEFAULT_TAU, accuracy: float = 0.80,
                   robust_thresh: float = 0.90, n_mc: int = 64, seed: int = 0,
                   sigma_grid: Optional[np.ndarray] = None) -> float:
    """Smallest weight-noise sigma at which fewer than `accuracy` of cases remain
    individually robust.

    Two-level definition: add N(0, sigma^2) to each weight (clipped to [0,1]); a
    *case* is robust at sigma if it keeps its unperturbed signed VERDICT
    (determinate-for / determinate-against / indeterminate) on at least
    `robust_thresh` of MC draws; sigma* is the smallest sigma at which the
    fraction of robust cases falls below `accuracy`.

    Verdict-stability (not just determinate/indeterminate class) so that a noise
    level that flips a determinate-for case to determinate-against counts as
    instability. The grid runs to CRITICAL_SIGMA_CEIL; a returned value equal to
    the ceiling is right-censored (true tolerance is >= ceiling) -- callers
    should flag these rather than average them as if measured.
    """
    rng = np.random.default_rng(seed)
    w = np.asarray(weights, float)
    A = np.asarray(activation_matrix, float)
    n_p = w.size
    if sigma_grid is None:
        sigma_grid = np.linspace(0.0, CRITICAL_SIGMA_CEIL, 51)

    true_F = A @ w
    true_v = np.sign(true_F); true_v[np.abs(true_F) <= tau] = 0    # (cases,) verdict
    noise = rng.standard_normal((n_mc, A.shape[0], n_p))          # (mc, cases, p)

    prev_sigma, prev_surv = 0.0, 1.0
    for sigma in sigma_grid:
        if sigma == 0:
            prev_sigma, prev_surv = 0.0, 1.0
            continue
        w_noisy = np.clip(w[None, None, :] + sigma * noise, 0.0, 1.0)
        F = np.einsum("mcp,cp->mc", w_noisy, A)                   # (mc, cases)
        v = np.sign(F); v[np.abs(F) <= tau] = 0
        agree_frac = np.mean(v == true_v[None, :], axis=0)        # per case
        survival = float(np.mean(agree_frac >= robust_thresh))
        if survival < accuracy:
            frac = (accuracy - prev_surv) / (survival - prev_surv)
            return float(prev_sigma + frac * (sigma - prev_sigma))
        prev_sigma, prev_surv = sigma, survival
    return float(sigma_grid[-1])  # right-censored at the ceiling


# ---------------------------------------------------------------------------
# Hart baselines  (R4 — "more law than Hart", reported as a margin/curve)
# ---------------------------------------------------------------------------
def hart_consensus_determinate(activations, frac: float = 0.6,
                               act_cut: float = 0.1) -> np.ndarray:
    """Consensus Hart (the faithful operationalization of Hart-as-validation):
    a case is determinate only when the share of *active* principles pointing the
    same direction exceeds `frac`. Hart requires agreement; below `frac` the law
    'runs out'."""
    A = np.atleast_2d(np.asarray(activations, float))
    pos = np.sum(A > act_cut, axis=1)
    neg = np.sum(A < -act_cut, axis=1)
    active = pos + neg
    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.maximum(pos, neg) / np.where(active == 0, 1, active)
    return (active > 0) & (share > frac)


def hart_hierarchical_determinate(weights, activations, tier_of: np.ndarray,
                                  frac: float = 0.6, act_cut: float = 0.1,
                                  tau: float = DEFAULT_TAU) -> np.ndarray:
    """Charitable *lexical-priority* Hart: principles sit in priority tiers
    (0 = top). For each case, the highest-priority tier that has ANY active
    principle is DISPOSITIVE: that tier decides iff its active principles reach
    consensus (>= `frac` agree in sign); otherwise the law runs out. Lower tiers
    are NOT consulted once a higher tier is reached — that is the lexical priority
    Berman's nonlexical model is contrasted against, and it is what stops Hart
    from finding law in nearly every case via tier escape-hatches.
    """
    A = np.atleast_2d(np.asarray(activations, float))
    tier_of = np.asarray(tier_of, int)
    out = np.zeros(A.shape[0], dtype=bool)
    reached = np.zeros(A.shape[0], dtype=bool)  # a dispositive tier has been reached
    for t in np.unique(tier_of):
        idx = np.where(tier_of == t)[0]
        sub = A[:, idx]
        pos = np.sum(sub > act_cut, axis=1)
        neg = np.sum(sub < -act_cut, axis=1)
        active = pos + neg
        this_tier = (active > 0) & (~reached)        # cases first resolved at this tier
        with np.errstate(invalid="ignore", divide="ignore"):
            share = np.maximum(pos, neg) / np.where(active == 0, 1, active)
        out |= this_tier & (share > frac)             # decided iff consensus here (strict, matches consensus Hart)
        reached |= this_tier                           # dispositive; no escape downward
    return out


def hart_validation_status(activations, k: int = 2, act_cut: float = 0.5) -> np.ndarray:
    """Naive validation foil used to make Berman's 'no contradiction' point.

    A direction counts as independently 'validated' when at least `k` principles
    are strongly activated (|a| > act_cut) in that direction. Returns one integer
    code per case:
        0 = gap          (neither direction validated -> the law runs out)
        1 = for          (only the positive direction validated)
        2 = against      (only the negative direction validated)
        3 = contradiction(both directions validated -> q and not-q are both law)

    The point: code 3 is possible for a validation model, but impossible for
    Berman's aggregation, because a single net force has a single sign. So the sum
    never returns code 3.
    """
    A = np.atleast_2d(np.asarray(activations, float))
    pos = np.sum(A > act_cut, axis=1) >= k
    neg = np.sum(A < -act_cut, axis=1) >= k
    code = np.zeros(A.shape[0], dtype=int)
    code[pos & ~neg] = 1
    code[neg & ~pos] = 2
    code[pos & neg] = 3
    return code


def hart_advantage(weights, activation_matrix, tier_of, frac: float = 0.6,
                   tau: float = DEFAULT_TAU) -> float:
    """Berman determinacy rate minus hierarchical-Hart determinacy rate (a margin,
    not a pass/fail)."""
    berman = determinacy_rate(weights, activation_matrix, tau)
    hart = float(np.mean(hart_hierarchical_determinate(
        weights, activation_matrix, tier_of, frac=frac, tau=tau)))
    return berman - hart


# ---------------------------------------------------------------------------
# R3 — nonlexicality.  Does determination track activation MAGNITUDES (and many
# principles jointly), as Berman requires, rather than just signs/ranks (lexical)?
# ---------------------------------------------------------------------------
def ordinalize(weights: np.ndarray) -> np.ndarray:
    """Replace weights by rank-based pseudo-weights (1..n)/n — discards cardinal info."""
    order = np.argsort(np.argsort(weights))  # ranks 0..n-1
    return (order + 1) / len(weights)


# Verdict functions return an integer verdict per case: +1 / -1 (determinate for
# / against) or 0 (indeterminate). These are the inputs to the R3 nonlexicality
# probe and are reused by the aggregation-function suite.
def saw_verdict(weights, activation_matrix, tau: float = DEFAULT_TAU) -> np.ndarray:
    """Nonlexical: verdict from the *magnitude-weighted sum* of all principles."""
    F = np.asarray(activation_matrix, float) @ np.asarray(weights, float)
    v = np.sign(F)
    v[np.abs(F) <= tau] = 0
    return v.astype(int)


def lexicographic_verdict(weights, activation_matrix, act_cut: float = 0.05) -> np.ndarray:
    """Lexical: verdict from the SIGN of the single highest-weight active
    principle (Raz exclusionary-reasons style). This is the mode Berman's theory
    is defined against — included only as the lexical foil for the R3 probe."""
    w = np.asarray(weights, float)
    A = np.atleast_2d(np.asarray(activation_matrix, float))
    order = np.argsort(-w)  # highest weight first
    out = np.zeros(A.shape[0], dtype=int)
    decided = np.zeros(A.shape[0], dtype=bool)
    for j in order:
        active = (np.abs(A[:, j]) > act_cut) & (~decided)
        out[active] = np.sign(A[active, j]).astype(int)
        decided |= active
    return out


def nonlexicality_score(verdict_fn: Callable[[np.ndarray], np.ndarray],
                        activation_matrix: np.ndarray,
                        seed: int = 0, n_mc: int = 32, jitter: float = 0.20) -> float:
    """R3 probe. Fraction of cases whose VERDICT changes under a magnitude-only
    perturbation of activations that preserves every activation's sign AND the
    within-case rank order of |activation|.

    A lexical rule (decides on the top-ranked active principle) is invariant to
    such perturbations -> score ~ 0. A genuinely nonlexical, magnitude-aggregating
    rule (SAW) is sensitive -> score > 0. `verdict_fn(A)` maps an (n,9) activation
    matrix to an integer verdict array in {-1,0,+1}.
    """
    rng = np.random.default_rng(seed)
    A = np.asarray(activation_matrix, float)
    base = verdict_fn(A)
    rank = np.argsort(np.argsort(np.abs(A), axis=1), axis=1)  # rank of |a| within row
    n_p = A.shape[1]
    changed = np.zeros(A.shape[0])
    for _ in range(n_mc):
        mag = np.clip(np.abs(A) * (1 + rng.uniform(-jitter, jitter, A.shape)), 0, 1)
        sorted_mag = np.sort(mag, axis=1)
        # Strictly-increasing epsilon breaks ties created when two magnitudes both
        # clip to 1.0; without this, re-placing tied values by index can silently
        # change the rank order (and thus the decision of a lexical rule). This
        # guarantees |A2| keeps the ORIGINAL within-row rank order exactly.
        sorted_mag = sorted_mag + 1e-9 * np.arange(n_p)
        A2 = np.sign(A) * np.take_along_axis(sorted_mag, rank, axis=1)
        changed += (verdict_fn(A2) != base)
    return float(np.mean(changed / n_mc))


# ---------------------------------------------------------------------------
# Descriptive weight statistics (D1 gradation, and emergence diagnostics)
# ---------------------------------------------------------------------------
def weight_cv(weights: np.ndarray) -> float:
    w = np.asarray(weights, float)
    m = w.mean()
    return float(w.std() / m) if m > 0 else 0.0


def n_survivors(weights: np.ndarray, cutoff: float = 0.05) -> int:
    return int(np.sum(np.asarray(weights, float) > cutoff))


def cluster_gap(weights: np.ndarray) -> float:
    """Gap between the two k-means clusters of weights (bimodality diagnostic)."""
    w = np.sort(np.asarray(weights, float))
    if w.size < 2:
        return 0.0
    best = None
    for k in range(1, w.size):
        lo, hi = w[:k], w[k:]
        gap = hi.mean() - lo.mean()
        if best is None or gap > best:
            best = gap
    return float(best)


# ===========================================================================
# Agent-based model of weight emergence (Notebook 1)
# ===========================================================================
# A bench of judges, each carrying an interpretive methodology that sets how
# often it invokes each principle. A principle's emergent "uptake share" is how
# often the bench takes it up across decisions. This is a deliberately simple
# model of Berman's "taking-up" (JESP p. 302); it is NOT his full account of
# weight, which is conferred by reliance and by winning conflicts (pp. 322, 324)
# and is taken up in Notebook 1c. The emergent quantity here is called "uptake
# share," never "weight in Berman's sense."

METHODOLOGIES = ("textualist", "purposivist", "pragmatist", "originalist")

# For each methodology, the principles it tends to invoke often ('high') or
# rarely ('low'); anything unlisted is 'medium'. Names refer to NAMED_PRINCIPLES.
_METH_SETS = {
    "textualist":  {"high": {"text", "original"},
                    "low":  {"dignity", "equality", "liberty", "welfare"}},
    "purposivist": {"high": {"dignity", "equality", "liberty"},
                    "low":  {"text", "original", "federalism"}},
    "pragmatist":  {"high": {"welfare", "precedent"},
                    "low":  {"dignity", "equality", "original"}},
    "originalist": {"high": {"original", "text", "federalism"},
                    "low":  {"dignity", "equality", "liberty", "welfare"}},
}

# Invocation-probability ranges (high, medium, low) by how specialized the
# methodologies are. 'wide' = strong specialization (headline); 'narrow' = nearly
# homogeneous (used to show heterogeneity is necessary for the split).
_SPREAD_RANGES = {
    "wide":   ((0.85, 0.95), (0.35, 0.65), (0.02, 0.08)),
    "medium": ((0.70, 0.80), (0.40, 0.60), (0.15, 0.25)),
    "narrow": ((0.60, 0.70), (0.42, 0.58), (0.30, 0.40)),
}


def methodology_tendencies(methodology: str, spread: str, rng) -> np.ndarray:
    """A 9-vector of per-principle invocation probabilities for one judge of a
    given methodology, drawn from the high/medium/low ranges set by `spread`."""
    hi, med, lo = _SPREAD_RANGES[spread]
    sets = _METH_SETS[methodology]
    t = np.empty(len(NAMED_PRINCIPLES))
    for i, (name, _) in enumerate(NAMED_PRINCIPLES):
        rng_range = hi if name in sets["high"] else (lo if name in sets["low"] else med)
        t[i] = rng.uniform(*rng_range)
    return t


class LegalSystem:
    """A bench of judges whose case-by-case invocations produce emergent
    principle uptake-shares.

    `turnover` selects how judges are replaced over time:
      'static'      no replacement (the headline emergence must hold here);
      'fitness'     replace the least decisive 20% with mutated copies of the
                    most decisive 20% (the old, suspect, non-Bermanian rule);
      'appointment' replace a random 20%, reseated from the methodology pool;
      'noise'       replace a random 20% with mutated copies of random judges.
    The point of the four regimes is to show the emergent structure does NOT
    depend on the turnover rule.
    """

    def __init__(self, n_agents=100, methodology_mix=None, spread="wide",
                 use_threshold=True, threshold_pct=50, turnover="static",
                 boost_cut=0.3, learn_from_system=0.15, seed=0):
        # One master seed -> five independent streams. No global np.random.
        ss = np.random.SeedSequence(seed)
        (self._rng_agents, self._rng_panels, self._rng_cases,
         self._rng_dec, self._rng_rep) = (np.random.default_rng(s) for s in ss.spawn(5))
        self.n_agents = n_agents
        self.spread = spread
        self.use_threshold = use_threshold
        self.threshold_pct = threshold_pct
        self.turnover = turnover
        self.boost_cut = boost_cut
        self.learn_from_system = learn_from_system
        self.n_p = len(NAMED_PRINCIPLES)

        # Assign methodologies exactly (round-robin or a given mix), then permute
        # the arrangement with the seeded generator (never np.random.shuffle).
        if methodology_mix is None:
            base = [METHODOLOGIES[i % 4] for i in range(n_agents)]
        else:
            base = []
            for m, frac in methodology_mix.items():
                base += [m] * int(round(frac * n_agents))
            base = (base + [METHODOLOGIES[0]] * n_agents)[:n_agents]
        self.methodology = list(self._rng_agents.permutation(base))
        self.tendencies = np.array([methodology_tendencies(m, spread, self._rng_agents)
                                    for m in self.methodology])     # (n_agents, 9)
        self.system_weights = np.full(self.n_p, 0.5)                # uniform init
        self.raw_uptake = np.full(self.n_p, 0.5)                    # pre-threshold share
        self.success = np.zeros(n_agents)
        self.decisions = np.zeros(n_agents)
        self.trajectory = []

    def _recompute_weights(self, counts, total):
        raw = counts / max(total, 1)
        raw = raw / max(raw.max(), 1e-12)                           # normalize to max
        self.raw_uptake = raw.copy()                               # the pre-threshold vector
        w = raw.copy()
        if self.use_threshold:                                     # zero the bottom percentile
            cut = np.percentile(w, self.threshold_pct)
            w = np.where(w < cut, 0.0, w)
            if w.max() > 0:
                w = w / w.max()
        self.system_weights = w

    def _replace(self, idx_out, idx_template):
        for o, t in zip(idx_out, idx_template):
            self.methodology[o] = self.methodology[t]
            self.tendencies[o] = np.clip(self.tendencies[t] + self._rng_rep.normal(0, 0.03, self.n_p),
                                         0.02, 0.98)
            self.success[o] = 0.0
            self.decisions[o] = 0.0

    def _turnover_step(self):
        k = max(1, int(0.2 * self.n_agents))
        if self.turnover == "static":
            return
        if self.turnover == "fitness":
            # eligible only with >= 5 decisions, so 1-2-case luck cannot rank an agent
            rate = np.where(self.decisions >= 5, self.success / np.maximum(self.decisions, 1), -1.0)
            order = np.argsort(rate)
            self._replace(order[:k], order[-k:])                   # bottom k <- top k
        elif self.turnover == "appointment":
            for o in self._rng_rep.choice(self.n_agents, k, replace=False):
                m = METHODOLOGIES[int(self._rng_rep.integers(4))]
                self.methodology[o] = m
                self.tendencies[o] = methodology_tendencies(m, self.spread, self._rng_rep)
                self.success[o] = 0.0
                self.decisions[o] = 0.0
        elif self.turnover == "noise":
            self._replace(self._rng_rep.choice(self.n_agents, k, replace=False),
                          self._rng_rep.choice(self.n_agents, k, replace=False))

    def simulate(self, n_cases=2000, panel_size=9, update_every=100, selection_every=500):
        counts = np.zeros(self.n_p)
        total = 0
        A = self._rng_cases.uniform(-1, 1, size=(n_cases, self.n_p))           # case stream up front
        panels = np.array([self._rng_panels.choice(self.n_agents, panel_size, replace=False)
                           for _ in range(n_cases)])
        self.trajectory = []
        for c in range(n_cases):
            a = A[c]
            panel = panels[c]
            tend = self.tendencies[panel]                                      # (panel, 9)
            prob = 0.85 * tend + self.learn_from_system * self.system_weights[None, :]
            boost = np.abs(a) > self.boost_cut                                 # strong activations invoked more
            prob = np.where(boost[None, :], np.minimum(prob * 1.15, 0.99), prob)
            invoked = self._rng_dec.uniform(size=(panel_size, self.n_p)) < prob
            counts += invoked.sum(axis=0)
            total += int(invoked.sum())
            forces = (invoked * self.system_weights[None, :] * a[None, :]).sum(axis=1)  # SAW per judge
            med = np.median(np.abs(forces))
            self.success[panel] += (np.abs(forces) >= med)                     # fitness bookkeeping
            self.decisions[panel] += 1
            if (c + 1) % update_every == 0:
                self._recompute_weights(counts, total)
                # record the smooth pre-threshold uptake for convergence/trajectory
                self.trajectory.append((c + 1, self.raw_uptake.copy()))
            if (c + 1) % selection_every == 0:
                self._turnover_step()
        self._recompute_weights(counts, total)
        return self.system_weights


def gmm_bimodal(values, seed=0, dbic_thresh=10.0):
    """Two-vs-one Gaussian-mixture BIC test for bimodality. Returns
    (dbic, separated, is_bimodal): dbic = BIC(1) - BIC(2) (larger favors two
    components); `separated` requires the two means be at least one larger-
    component SD apart (rejects two overlapping Gaussians). Deterministic."""
    from sklearn.mixture import GaussianMixture
    x = np.asarray(values, float).reshape(-1, 1)
    if len(np.unique(np.round(x, 6))) < 2:
        return 0.0, False, False
    g1 = GaussianMixture(1, random_state=seed, n_init=5).fit(x)
    g2 = GaussianMixture(2, random_state=seed, n_init=5).fit(x)
    dbic = float(g1.bic(x) - g2.bic(x))
    means = g2.means_.ravel()
    sds = np.sqrt(g2.covariances_.ravel())
    separated = bool(abs(means[0] - means[1]) > sds.max())
    return dbic, separated, bool(dbic > dbic_thresh and separated)


def bimodality_coefficient(values):
    """Sarle's bimodality coefficient. > 0.555 suggests bimodality. Underpowered
    for very small n, so use it on pooled data, not a single 9-vector."""
    from scipy import stats as _st
    x = np.asarray(values, float)
    n = x.size
    if n < 4:
        return float("nan")
    g = _st.skew(x); k = _st.kurtosis(x, fisher=True)
    return float((g**2 + 1) / (k + 3 * (n - 1)**2 / ((n - 2) * (n - 3))))


# ===========================================================================
# Reliance / by-battle weight conferral (Notebook 2)
# ===========================================================================
# Berman's conferred weight (JESP pp. 322, 324): a principle gains weight by
# winning CONTESTED, DISPOSITIVE battles, amplified by how much the deciding
# precedent is later RELIED UPON (cited), decayed gradually, and removed when a
# precedent is OVERRULED. Battles are decided by a FIXED exogenous "pressure"
# structure (the practice), NOT by the conferred weight being built: making it
# literally circular ("weight made so by battle") produces runaway lock-in, so
# the decision and the read-out are decoupled. There is NO weight floor -- that
# is what made the old "precedent stock" collapse to uniform weights.

def weight_entropy(w):
    """Shannon entropy of a normalized weight vector (higher = more spread)."""
    w = np.asarray(w, float); s = w.sum()
    if s <= 0:
        return 0.0
    p = w / s; p = p[p > 0]
    return float(-(p * np.log(p)).sum())


def gini(w):
    """Gini coefficient of a weight vector (0 = equal, ->1 = concentrated)."""
    w = np.sort(np.asarray(w, float)); n = len(w)
    if w.sum() == 0:
        return 0.0
    return float((2 * np.arange(1, n + 1) - n - 1).dot(w) / (n * w.sum()))


def n_bins_occupied(w, edges=(0.2, 0.4, 0.6, 0.8)):
    """How many of Berman's ~5 rough weight bands a weight vector occupies."""
    return int(len(np.unique(np.digitize(np.asarray(w, float), edges))))


class RelianceSystem:
    """Conferred weight as reliance-amplified, decayed, overruling-prunable
    battle-victories. Battles are decided by `pressure_weights` (fixed), and the
    conferred weight is read out separately, so the dynamics cannot lock in."""

    def __init__(self, pressure_weights=None, invoke_prob=None, salience=None,
                 activations="correlated", cite_mode="similarity", n_cite=3,
                 overrule_rate=0.02, decay_rate=0.9995, clash_cut=0.10,
                 feedback_alpha=0.0, tau=DEFAULT_TAU, seed=0):
        # feedback_alpha > 0 feeds the running conferred weight back into the
        # battle decision (the literally-circular "made so by battle"). It exists
        # only to DEMONSTRATE the runaway lock-in that justifies the decoupled
        # design; the headline runs use feedback_alpha = 0.
        ss = np.random.SeedSequence(seed)
        (self._rng_cases, self._rng_cite, self._rng_over,
         self._rng_inv, self._rng_setup) = (np.random.default_rng(s) for s in ss.spawn(5))
        n_p = len(NAMED_PRINCIPLES)
        self.n_p = n_p
        self.pressure_weights = (named_weight_vector("high", seed) if pressure_weights is None
                                 else np.asarray(pressure_weights, float))
        # uptake invocation propensity. It tracks the same exogenous PRESSURE
        # structure that decides battles (a principle pressed harder is invoked
        # more and can win more), so uptake and conferred weight are correlated
        # but distinct. It is NOT a function of the conferred weight being built,
        # so the two can still come apart (the forced-loser demo overrides it).
        self.invoke_prob = (np.clip(0.2 + 0.7 * (self.pressure_weights / self.pressure_weights.max())
                                    + self._rng_setup.normal(0, 0.05, n_p), 0.05, 0.98)
                            if invoke_prob is None else np.asarray(invoke_prob, float))
        self.salience = np.ones(n_p) if salience is None else np.asarray(salience, float)
        self.activations, self.cite_mode, self.n_cite = activations, cite_mode, n_cite
        self.overrule_rate, self.decay_rate, self.clash_cut, self.tau = \
            overrule_rate, decay_rate, clash_cut, tau
        self.feedback_alpha = feedback_alpha
        self.invoked_total = np.zeros(n_p)
        self.n_seen = 0
        self._ns = 0

    def run(self, n_cases=2000):
        np_ = self.n_p
        A = (category_correlated_activations(n_cases, self._rng_cases) if self.activations == "correlated"
             else random_activations(n_cases, np_, self._rng_cases))
        A = np.clip(A * self.salience[None, :], -1, 1)
        inv = self._rng_inv.uniform(size=(n_cases, np_))
        act = np.zeros((n_cases, np_)); winners = np.zeros((n_cases, np_), bool)
        verdict = np.zeros(n_cases); margin = np.zeros(n_cases); opposing = np.zeros(n_cases)
        contested = np.zeros(n_cases, bool); cc = np.zeros(n_cases)
        good = np.zeros(n_cases, bool); born = np.zeros(n_cases, int); nrm = np.zeros(n_cases)
        ns = 0
        eff = self.pressure_weights.copy()                        # effective decision pressure
        for c in range(n_cases):
            a = A[c]
            self.invoked_total += (inv[c] < self.invoke_prob)     # uptake (weight-independent)
            self.n_seen += 1
            contrib = eff * a
            F = contrib.sum()
            if abs(F) <= self.tau:
                continue                                          # equipoise: no victor
            v = 1.0 if F > 0 else -1.0
            big = np.abs(contrib) > self.clash_cut
            win = (np.sign(contrib) == v) & big                  # dispositive winners only
            cont = bool((contrib > self.clash_cut).any() and (contrib < -self.clash_cut).any())
            opp = float(np.abs(contrib[(np.sign(contrib) != v) & big]).sum())
            act[ns] = a; winners[ns] = win; verdict[ns] = v; margin[ns] = abs(F)
            opposing[ns] = opp; contested[ns] = cont; born[ns] = c
            nrm[ns] = np.linalg.norm(a); good[ns] = True
            if ns > 0:                                           # reliance: cite prior good precedents
                gi = np.where(good[:ns])[0]
                if len(gi):
                    if self.cite_mode == "preferential":
                        order = gi[np.argsort(-cc[gi])]
                    elif self.cite_mode == "random":
                        order = self._rng_cite.permutation(gi)
                    else:                                        # similarity / blended (cosine of facts)
                        sims = (act[gi] @ a) / (nrm[gi] * np.linalg.norm(a) + 1e-9)
                        score = sims * (1 + cc[gi]) if self.cite_mode == "blended" else sims
                        order = gi[np.argsort(-score)]
                    cc[order[:self.n_cite]] += 1
                    if self._rng_over.uniform() < self.overrule_rate:   # overrule a contradictory good precedent
                        sims = (act[gi] @ a) / (nrm[gi] * np.linalg.norm(a) + 1e-9)
                        cand = gi[(verdict[gi] == -v) & (sims > 0.5)]
                        if len(cand):
                            good[cand[0]] = False
            ns += 1
            if self.feedback_alpha > 0 and (c + 1) % 100 == 0 and ns > 0:
                mm = good[:ns] & contested[:ns]                   # lock-in demo: feed weight back into decisions
                if mm.any():
                    dec = self.decay_rate ** ((c + 1) - born[:ns][mm])
                    eq = opposing[:ns][mm] / (margin[:ns][mm] + opposing[:ns][mm] + 1e-9)
                    amp = margin[:ns][mm] * (1 + cc[:ns][mm]) * eq * dec
                    cur = (amp[:, None] * winners[:ns][mm]).sum(axis=0)
                    if cur.max() > 0:
                        eff = (1 - self.feedback_alpha) * self.pressure_weights + \
                              self.feedback_alpha * (cur / cur.max())
        self._act, self._winners, self._verdict = act[:ns], winners[:ns], verdict[:ns]
        self._margin, self._opposing, self._contested = margin[:ns], opposing[:ns], contested[:ns]
        self._cc, self._good, self._born = cc[:ns], good[:ns], born[:ns]
        self._n_cases, self._ns = n_cases, ns
        return self.read_weights()

    def _readout(self, use_reliance=True, use_equipoise=True):
        if self._ns == 0:
            return np.zeros(self.n_p)
        m = self._good & self._contested
        if not m.any():
            return np.zeros(self.n_p)
        decay = self.decay_rate ** (self._n_cases - self._born[m])
        equ = (self._opposing[m] / (self._margin[m] + self._opposing[m] + 1e-9)) if use_equipoise else 1.0
        rel = (1 + self._cc[m]) if use_reliance else 1.0
        amp = self._margin[m] * rel * equ * decay
        conf = (amp[:, None] * self._winners[m]).sum(axis=0)
        mx = conf.max()
        return conf / mx if mx > 0 else conf

    def read_weights(self):
        return self._readout(True, True)

    def read_weights_no_reliance(self):
        return self._readout(False, True)

    @property
    def uptake_share(self):
        return self.invoked_total / max(self.n_seen, 1)


if __name__ == "__main__":
    # quick self-test: static properties
    rng = np.random.default_rng(0)
    w = named_weight_vector("moderate", seed=0)
    A = random_activations(2000, 9, rng)
    print("categories:", CATEGORY_INDEX)
    print("determinacy rate (tau=0.2):", round(determinacy_rate(w, A), 4))
    print("critical sigma:", round(critical_sigma(w, A), 4))
    print("nonlexicality SAW vs lexical:",
          round(nonlexicality_score(lambda X: saw_verdict(w, X), A), 4),
          round(nonlexicality_score(lambda X: lexicographic_verdict(w, X), A), 4))
    # quick self-test: ABM emergence
    sysm = LegalSystem(turnover="static", seed=0)
    we = sysm.simulate(n_cases=2000)
    print("\nABM emergent uptake (post-threshold):", np.round(we, 2))
    print("ABM raw uptake (pre-threshold):", np.round(sysm.raw_uptake, 2))
    print("emerged (!= uniform):", not np.allclose(sysm.raw_uptake, 0.5))
    print("raw-uptake bimodal (GMM):", gmm_bimodal(sysm.raw_uptake))
    print("cluster gap:", round(cluster_gap(sysm.raw_uptake), 3), "| CV:", round(weight_cv(sysm.raw_uptake), 3))


Overwriting berman_core.py


In [3]:
# Reference values for the self-check, set from the first verified run.
REF_GAP_STATIC=0.461; REF_GAP_FITNESS=0.664; REF_POOLED_DBIC=664.2; REF_SIGMA_STATIC=0.096; REF_MARGIN_STATIC=0.271

In [4]:
import os, json
import numpy as np, pandas as pd
from scipy import stats
from berman_core import *

OUTDIR = os.environ.get('BERMAN_OUTDIR', 'data/notebook1')
os.makedirs(OUTDIR, exist_ok=True)
summary = {}

REGIMES = ['static', 'appointment', 'noise', 'fitness']   # static first: zero selection (the cleanest test)
SEEDS, N_CASES = 30, 2000
PRINCIPLE_NAMES = [n for n, _ in NAMED_PRINCIPLES]
print('Core + ABM loaded. Principles:', PRINCIPLE_NAMES)

Core + ABM loaded. Principles: ['text', 'original', 'precedent', 'structure', 'federalism', 'dignity', 'equality', 'liberty', 'welfare']


## Experiment 1 - Emergence, and whether it depends on the replacement rule

Every judge starts from a uniform weight of 0.5; only the methodologies differ.
We run 2,000 cases and read off the emergent uptake share for each principle.

We run this under four rules for replacing judges over time, holding everything
else fixed and pairing the random seeds across rules: **static** (no
replacement), **appointment** (retire a random fifth, reseat from the same
methodology pool), **noise** (replace a random fifth with copies of random
judges), and **fitness** (replace the least decisive fifth with copies of the
most decisive). Fitness is the rule the old paper used; it is not Berman's
mechanism, so the test is whether the result needs it.

We report three things: whether a stable vector emerges, whether it separates
into a high and a low cluster (the legal/sub-legal boundary), and whether that
separation depends on the replacement rule. Bimodality is read off the
pre-threshold uptake share. At nine principles a single run is too small for a
mixture test to resolve two components, so the per-run measure is the cluster
gap, and the formal bimodality test is run on the pooled distribution across
seeds.

In [5]:
def live_set(w):
    return tuple(int(w[i] > 0.05) for i in range(9))   # which principles survive the threshold

EMERGENT = {}   # (regime, seed) -> {'w': post-threshold weights, 'raw': pre-threshold uptake}
rows = []
for reg in REGIMES:
    for s in range(SEEDS):
        sysm = LegalSystem(turnover=reg, use_threshold=True, threshold_pct=50, spread='wide', seed=s)
        w = sysm.simulate(n_cases=N_CASES)             # post-threshold emergent weights
        raw = sysm.raw_uptake                          # pre-threshold uptake share
        EMERGENT[(reg, s)] = dict(w=w.copy(), raw=raw.copy())
        traj = sysm.trajectory
        delta = float(np.mean(np.abs(traj[-1][1] - traj[-2][1]))) if len(traj) >= 2 else np.nan
        rec = dict(regime=reg, seed=s, cluster_gap=cluster_gap(raw), weight_cv=weight_cv(raw),
                   n_live=n_survivors(w, 0.05), gmm_dbic=gmm_bimodal(raw)[0], delta_last=delta)
        for i, nm in enumerate(PRINCIPLE_NAMES):
            rec['raw_' + nm] = raw[i]
        rows.append(rec)
df1 = pd.DataFrame(rows); df1.to_csv(f'{OUTDIR}/exp1_emergence.csv', index=False)

# emergence: did uptake move off the uniform 0.5 start?
raw_all = np.array([EMERGENT[(r, s)]['raw'] for r in REGIMES for s in range(SEEDS)])
emerged = bool(np.mean(np.abs(raw_all.mean(0) - 0.5)) > 0.05)

# membership: is the live/dead set the same across seeds, and across rules?
mem = {reg: set(live_set(EMERGENT[(reg, s)]['w']) for s in range(SEEDS)) for reg in REGIMES}
membership_stable = all(len(mem[reg]) == 1 for reg in REGIMES)
membership_same = membership_stable and len({next(iter(mem[reg])) for reg in REGIMES}) == 1
frac_live = (raw_all > raw_all.mean(1, keepdims=True)).mean(0)
always_live = [PRINCIPLE_NAMES[i] for i in range(9) if frac_live[i] > 0.99]
always_dead = [PRINCIPLE_NAMES[i] for i in range(9) if frac_live[i] < 0.01]

# bimodality: pooled across all runs (the formal test); per-seed dbic for context
pooled = raw_all.ravel()
db_pool, sep_pool, bim_pool = gmm_bimodal(pooled)
bc_pool = bimodality_coefficient(pooled)
per_seed_bimodal = int((df1.gmm_dbic > 10).sum())

# gap by rule, and whether the rules differ (seed-paired Wilcoxon vs static)
gap_by_reg = df1.groupby('regime').cluster_gap.mean()
def paired(a, b):
    ga = df1[df1.regime == a].sort_values('seed').cluster_gap.values
    gb = df1[df1.regime == b].sort_values('seed').cluster_gap.values
    return float(stats.wilcoxon(ga, gb).pvalue)
p_static_appt = paired('static', 'appointment')
p_static_noise = paired('static', 'noise')
p_static_fit = paired('static', 'fitness')

summary['exp1'] = dict(emerged=emerged, membership_stable=membership_stable, membership_same_across_rules=membership_same,
    always_live=always_live, always_dead=always_dead, mean_cluster_gap=float(df1.cluster_gap.mean()),
    gap_by_regime={k: float(v) for k, v in gap_by_reg.items()}, per_seed_bimodal_of_120=per_seed_bimodal,
    pooled_gmm_dbic=db_pool, pooled_gmm_bimodal=bool(bim_pool), pooled_bc=bc_pool,
    p_static_vs_appointment=p_static_appt, p_static_vs_noise=p_static_noise, p_static_vs_fitness=p_static_fit)
print('emerged (moved off uniform):', emerged)
print('live cluster (always):', always_live)
print('dead cluster (always):', always_dead)
print('membership identical across seeds AND rules:', membership_same)
print('mean cluster gap by rule:', {k: round(v, 3) for k, v in gap_by_reg.items()})
print(f'per-seed GMM bimodal: {per_seed_bimodal}/120 (n=9 is underpowered); POOLED GMM dbic={db_pool:.1f} bimodal={bim_pool}, BC={bc_pool:.3f}')
print(f'gap static vs appointment p={p_static_appt:.2f}, vs noise p={p_static_noise:.2f}, vs fitness p={p_static_fit:.3f}')

emerged (moved off uniform): True
live cluster (always): ['text', 'original', 'precedent', 'structure', 'federalism']
dead cluster (always): ['dignity', 'equality', 'liberty', 'welfare']
membership identical across seeds AND rules: True
mean cluster gap by rule: {'appointment': 0.472, 'fitness': 0.663, 'noise': 0.452, 'static': 0.461}
per-seed GMM bimodal: 3/120 (n=9 is underpowered); POOLED GMM dbic=664.1 bimodal=True, BC=0.657
gap static vs appointment p=0.04, vs noise p=0.18, vs fitness p=0.000


*Reading.* Uptake moves off the uniform start and settles into the same high and
low groups every time. The low group is the moral principles (dignity, equality,
liberty, welfare): only the purposivist judges, a quarter of the bench, invoke
them often, while the other three methodologies invoke them rarely, so they fall
below the median. The high group is everything else (text, original, precedent,
structure, federalism): each is invoked at a high or at least middling rate by
most methodologies, so none falls below the line. The driver is which principles
a majority of methodologies are willing to take up, not one methodology's
preference: precedent has the highest uptake although only the pragmatists favour
it, and structure survives although no methodology favours it, because neither is
strongly disfavoured by anyone. The membership is identical across all 30 seeds
and all four replacement rules, and the separation is present under the static
rule, where no judge is ever replaced, so the boundary comes from the bench's
heterogeneity and the uptake threshold, not from any selection rule. The
non-fitness rules give nearly the same gap (static 0.461, appointment 0.472,
noise 0.452, a spread of 0.020; appointment differs from static at about p = 0.04
but by only 0.011 in the gap, which is trivial). Fitness is the only rule that
materially changes the gap, raising it to 0.66. The pooled mixture test confirms
two clusters at the population level; the clean dead-at-zero binary is then
produced by the uptake threshold, which sharpens the low group rather than
inventing it.

## Experiment 2 - Locating the cause: heterogeneity and threshold, not turnover

Experiment 1 showed the separation does not depend on the replacement rule. This
experiment shows what it does depend on. We move three knobs one at a time and
watch the cluster gap.

(a) **Methodological spread.** Make the methodologies more or less specialized.
The prediction is that the gap grows with specialization and collapses toward
zero when all methodologies invoke all principles at similar rates.

(b) **Uptake threshold.** Move the percentile at which a principle is counted as
dead. The prediction is that the size of the dead cluster moves with the
threshold but the underlying pre-threshold gap does not, confirming the threshold
sets the boundary rather than creating the separation.

(c) **Turnover cadence.** Change how often judges are replaced under the
appointment rule. The prediction is a flat gap, the negative control for
Experiment 1.

(d) **Bench composition.** Change the mix of methodologies, holding everything
else fixed, with the live/dead cut ON so there is a line to move. The prediction
is that the MEMBERSHIP of the two groups tracks the composition while the
two-group shape survives. This is the one knob the manuscript's claim that "who
sits on the bench decides which principles carry weight" actually rests on;
(a) varies specialization, not composition, and cannot support it.

In [6]:
# (a) methodological spread, static bench, pre-threshold gap
r2a = []
for spread in ['narrow', 'medium', 'wide']:
    for s in range(10):
        m = LegalSystem(turnover='static', use_threshold=False, spread=spread, seed=s); m.simulate(N_CASES)
        r2a.append(dict(spread=spread, seed=s, gap=cluster_gap(m.raw_uptake)))
df2a = pd.DataFrame(r2a)
gap_spread = df2a.groupby('spread').gap.mean()

# (b) uptake-threshold percentile: dead-cluster size moves, pre-threshold gap does not
r2b = []
for pct in [25, 33, 50, 67]:
    for s in range(10):
        m = LegalSystem(turnover='static', use_threshold=True, threshold_pct=pct, spread='wide', seed=s)
        w = m.simulate(N_CASES)
        r2b.append(dict(pct=pct, seed=s, n_live=n_survivors(w, 0.05), raw_gap=cluster_gap(m.raw_uptake)))
df2b = pd.DataFrame(r2b)

# (c) turnover cadence under appointment: flat gap (control)
r2c = []
for cad in [250, 500, 1000]:
    for s in range(10):
        m = LegalSystem(turnover='appointment', use_threshold=False, spread='wide', seed=s)
        m.simulate(N_CASES, selection_every=cad)
        r2c.append(dict(cadence=cad, seed=s, gap=cluster_gap(m.raw_uptake)))
df2c = pd.DataFrame(r2c)
# (d) bench composition, static bench, live/dead cut ON -> does the LINE move?
# (a) varies how specialized each methodology is; it does NOT vary the mix, and it
# runs with the cut off, so there is no live/dead line in it at all. Only this part
# can support the claim that changing who sits on the bench changes which principles
# carry weight.
r2d = []
MIXES = {
    'equal':             dict(textualist=.25, purposivist=.25, pragmatist=.25, originalist=.25),
    'textualist_heavy':  dict(textualist=.55, purposivist=.15, pragmatist=.15, originalist=.15),
    'purposivist_heavy': dict(textualist=.15, purposivist=.55, pragmatist=.15, originalist=.15),
}
PN = [p for p, _ in NAMED_PRINCIPLES]
for mixname, mix in MIXES.items():
    for s in range(SEEDS):
        m = LegalSystem(turnover='static', use_threshold=True, threshold_pct=50,
                        spread='wide', methodology_mix=mix, seed=s)
        w = m.simulate(N_CASES)
        live = tuple(PN[i] for i in range(9) if w[i] > 0.05)
        rec = dict(mix=mixname, seed=s, raw_gap=cluster_gap(m.raw_uptake),
                   n_live=n_survivors(w, 0.05), live_set='|'.join(live))
        for i, nm_ in enumerate(PN):
            rec['raw_' + nm_] = float(m.raw_uptake[i])
        r2d.append(rec)
df2d = pd.DataFrame(r2d)

for d, nm in [(df2a, 'exp2a_spread'), (df2b, 'exp2b_threshold'), (df2c, 'exp2c_cadence'), (df2d, 'exp2d_composition')]:
    d.to_csv(f'{OUTDIR}/{nm}.csv', index=False)
summary['exp2'] = dict(gap_by_spread={k: float(v) for k, v in gap_spread.items()},
    nlive_by_threshold={int(p): float(df2b[df2b.pct == p].n_live.mean()) for p in [25, 33, 50, 67]},
    rawgap_by_threshold={int(p): float(df2b[df2b.pct == p].raw_gap.mean()) for p in [25, 33, 50, 67]},
    gap_by_cadence={int(c): float(df2c[df2c.cadence == c].gap.mean()) for c in [250, 500, 1000]},
    composition={k: dict(raw_gap=float(df2d[df2d['mix'] == k].raw_gap.mean()),
                         live_sets={ls: int(n) for ls, n in df2d[df2d['mix'] == k].live_set.value_counts().items()})
                 for k in MIXES})
print('(a) gap by spread:', {k: round(v, 3) for k, v in gap_spread.items()}, '-> grows with specialization')
print('(b) live count by threshold pct:', summary['exp2']['nlive_by_threshold'], '| pre-threshold gap (constant):', {k: round(v,3) for k,v in summary['exp2']['rawgap_by_threshold'].items()})
print('(c) gap by turnover cadence:', {k: round(v, 3) for k, v in summary['exp2']['gap_by_cadence'].items()}, '-> flat')
print('(d) composition, raw gap:', {k: round(v['raw_gap'], 3) for k, v in summary['exp2']['composition'].items()})
for k in MIXES:
    print('   ', k, summary['exp2']['composition'][k]['live_sets'])


(a) gap by spread: {'medium': 0.257, 'narrow': 0.135, 'wide': 0.361} -> grows with specialization
(b) live count by threshold pct: {25: 7.0, 33: 6.0, 50: 5.0, 67: 3.0} | pre-threshold gap (constant): {25: 0.463, 33: 0.445, 50: 0.459, 67: 0.44}
(c) gap by turnover cadence: {250: 0.372, 500: 0.371, 1000: 0.366} -> flat
(d) composition, raw gap: {'equal': 0.461, 'textualist_heavy': 0.585, 'purposivist_heavy': 0.444}
    equal {'text|original|precedent|structure|federalism': 30}
    textualist_heavy {'text|original|precedent|structure|federalism': 30}
    purposivist_heavy {'precedent|structure|dignity|equality|liberty': 30}


*Reading.* The gap grows with methodological specialization (from about 0.14 for
a near-homogeneous bench to about 0.36 for a strongly specialized one), so
heterogeneity is what produces the separation. Moving the threshold changes how
many principles are counted as dead but leaves the pre-threshold gap unchanged,
so the threshold sets where the line falls rather than creating the two groups.
The live/dead membership is fixed at the fiftieth percentile used here; if the
threshold is pushed nearer the boundary, which principle drops out becomes
seed-dependent even though the separation itself does not. Turnover cadence does
nothing, confirming Experiment 1. Composition, by contrast, moves the membership: a
bench where the purposivist methodology holds a majority rather than a quarter
trades three principles for three, and the two-group shape survives the trade.
So spread governs how far apart the groups sit and composition governs who is in
them.


## Experiment 3 - Do the emergent weights satisfy what Berman requires?

This is the point of the notebook. We take the emergent weights from Experiment 1
(after running the model, not the uniform starting values) and put them through
the same requirements Notebook 0 used, on fresh cases:

- **R1 well-definedness.** The weights are nondegenerate: at least two survive,
  and the surviving set is stable across seeds. This is the requirement Notebook
  0 deferred, because it needs learned weights to test.
- **R2 rough weights suffice.** The emergent weights tolerate at least as much
  noise as the hand-set weights did (critical sigma in Notebook 0's range).
- **R3 nonlexicality.** Verdicts under the emergent weights still respond to
  activation magnitude far more than a lexical rule does.
- **R4 more law than Hart.** The emergent weights still beat a lexical-priority
  Hart (positive margin).
- **D1 gradation (descriptive only).** We report the within-cluster spread, but
  it never fails the theory: Berman permits equal weights among live principles.

We run this for all four replacement rules to show the properties, like the
separation, do not depend on the rule. One terminology note: these emergent
uptake shares are placed in the SAW weight slot to test R1-R4; they are not
Berman's conferred weight, which depends on a principle winning conflicts and
being relied upon, and which is left to Notebook 1c.

In [7]:
tier_of = np.array([{'textual': 0, 'structural': 1, 'moral': 2}[c] for _, c in NAMED_PRINCIPLES])
rows = []
for reg in REGIMES:
    for s in range(SEEDS):
        w = EMERGENT[(reg, s)]['w']                         # emergent weights, post-threshold
        A = random_activations(800, 9, np.random.default_rng(9000 + s))   # fresh held-out cases
        sig = critical_sigma(w, A, n_mc=40, seed=s)
        nl_saw = nonlexicality_score(lambda X: saw_verdict(w, X), A, seed=s)
        nl_lex = nonlexicality_score(lambda X: lexicographic_verdict(w, X), A, seed=s)
        margin = hart_advantage(w, A, tier_of)
        rows.append(dict(regime=reg, seed=s, sigma=sig, censored=int(sig >= CRITICAL_SIGMA_CEIL - 1e-9),
            nl_saw=nl_saw, nl_lex=nl_lex, hart_margin=margin, n_live=n_survivors(w, 0.05),
            weight_cv=weight_cv(w), cluster_gap=cluster_gap(w),
            R1=int(n_survivors(w, 0.05) >= 2), R2=int(sig >= 0.07),
            R3=int(nl_saw > 5 * max(nl_lex, 1e-9)), R4=int(margin > 0)))
df3 = pd.DataFrame(rows); df3.to_csv(f'{OUTDIR}/exp3_crossval.csv', index=False)
by = df3.groupby('regime')
summary['exp3'] = {reg: dict(R1=float(g.R1.mean()), R2=float(g.R2.mean()), R3=float(g.R3.mean()),
    R4=float(g.R4.mean()), mean_sigma=float(g.sigma.mean()), mean_margin=float(g.hart_margin.mean()),
    mean_cv=float(g.weight_cv.mean())) for reg, g in by}
print('Pass rates of the emergent weights (fraction of seeds), by replacement rule:')
print(f"{'rule':12s} {'R1':>5s} {'R2':>5s} {'R3':>5s} {'R4':>5s}  mean_sigma  mean_margin  weight_cv(D1)")
for reg in REGIMES:
    e = summary['exp3'][reg]
    print(f"{reg:12s} {e['R1']:5.2f} {e['R2']:5.2f} {e['R3']:5.2f} {e['R4']:5.2f}     {e['mean_sigma']:.3f}      {e['mean_margin']:+.3f}      {e['mean_cv']:.3f}")

Pass rates of the emergent weights (fraction of seeds), by replacement rule:
rule            R1    R2    R3    R4  mean_sigma  mean_margin  weight_cv(D1)
static        1.00  1.00  1.00  1.00     0.096      +0.271      0.903
appointment   1.00  1.00  1.00  1.00     0.097      +0.271      0.903
noise         1.00  1.00  1.00  1.00     0.095      +0.269      0.904
fitness       1.00  1.00  1.00  1.00     0.092      +0.267      0.906


*Reading.* The emergent weights satisfy R1-R4 under every replacement rule. They
are well-defined and stable (R1); they tolerate weight noise about as well as the
hand-set weights did (R2: mean critical sigma about 0.095 against Notebook 0's
0.090, though the worst single seed clears the 0.07 floor only narrowly, near
0.075); they keep determination magnitude-sensitive rather than lexical (R3); and
they still beat a lexical-priority Hart (R4). D1 (weight spread) is reported but
never used to pass or fail, because Berman permits equal weights among the live
principles.

## Experiment 4 - Is the bench actually doing weighted aggregation?

The cross-validation above assumes the bench's behavior is described by the SAW
formula with the emergent weights. This experiment checks that assumption. Each
judge sums only the principles it invokes, so the panel's mean force is a
masked version of the full weighted sum. We compare the panel's mean force on
fresh cases against the full SAW force `net_force(emergent weights, activations)`.

A high correlation alone would be near-automatic, because the two differ only by
the invocation mask. The informative number is the **verdict agreement**: on what
fraction of cases does the sign-with-threshold of the panel force match the SAW
verdict, at the same threshold. If invocation gating shrinks the forces, the two
can disagree near the threshold, and that is a real finding rather than something
to hide.

In [8]:
rows = []
for s in range(10):
    m = LegalSystem(turnover='static', use_threshold=True, spread='wide', seed=s)
    w = m.simulate(N_CASES)
    rng = np.random.default_rng(9500 + s)
    A = rng.uniform(-1, 1, (1000, 9))
    abm, saw = [], []
    for a in A:
        panel = rng.choice(m.n_agents, 9, replace=False)
        prob = 0.85 * m.tendencies[panel] + m.learn_from_system * w[None, :]
        prob = np.where((np.abs(a) > m.boost_cut)[None, :], np.minimum(prob * 1.15, 0.99), prob)
        invoked = rng.uniform(size=(9, 9)) < prob
        abm.append(float((invoked * w[None, :] * a[None, :]).sum(axis=1).mean()))   # panel mean force
        saw.append(float(net_force(w, a)))
    abm, saw = np.array(abm), np.array(saw)
    abm_v = np.sign(abm) * (np.abs(abm) > DEFAULT_TAU)
    agree = float(np.mean(abm_v == saw_verdict(w, A)))
    rows.append(dict(seed=s, r=float(np.corrcoef(abm, saw)[0, 1]),
                     rmse=float(np.sqrt(np.mean((abm - saw) ** 2))), verdict_agree=agree))
df4 = pd.DataFrame(rows); df4.to_csv(f'{OUTDIR}/exp4_saw_fidelity.csv', index=False)
summary['exp4'] = dict(mean_r=float(df4.r.mean()), mean_rmse=float(df4.rmse.mean()), mean_verdict_agree=float(df4.verdict_agree.mean()))
print(f"SAW fidelity: r={df4.r.mean():.3f}, RMSE={df4.rmse.mean():.3f}, verdict agreement={df4.verdict_agree.mean():.3f}")

SAW fidelity: r=0.970, RMSE=0.422, verdict agreement=0.859


*Reading.* The panel's behavior tracks the SAW formula closely in shape. The
verdict agreement reports how often the bench reaches the same determinate answer
as the formula at the same threshold; where it falls short, the cause is
invocation gating shrinking the forces, which we name rather than gloss. The
bench is doing weighted aggregation with the emergent weights, so the
cross-validation above is testing the right model.

## Experiment 5 - Co-invocation structure (descriptive)

A short consistency check. On the trained bench, do principles in the same
category get invoked together more than principles in different categories? This
connects to the correlation finding in Notebook 0. It is descriptive: the
tendency to co-invoke is partly built into the methodology table, so this checks
consistency, it is not an independent emergence result. We run it after the model
has run, on the trained bench (the old notebook measured untrained judges here,
which was a bug).

In [9]:
rows = []
for s in range(10):
    m = LegalSystem(turnover='static', use_threshold=True, spread='wide', seed=s)
    w = m.simulate(N_CASES)
    rng = np.random.default_rng(9700 + s)
    co = np.zeros((9, 9)); cnt = np.zeros(9)
    for _ in range(1000):
        a = rng.uniform(-1, 1, 9); panel = rng.choice(m.n_agents, 9, replace=False)
        prob = 0.85 * m.tendencies[panel] + m.learn_from_system * w[None, :]
        prob = np.where((np.abs(a) > m.boost_cut)[None, :], np.minimum(prob * 1.15, 0.99), prob)
        inv = (rng.uniform(size=(9, 9)) < prob).astype(float)
        co += inv.T @ inv; cnt += inv.sum(0)              # co-invocation counts (vectorized)
    norm = co / (np.sqrt(np.outer(cnt, cnt)) + 1e-9)
    T, S, M = CATEGORY_INDEX['textual'], CATEGORY_INDEX['structural'], CATEGORY_INDEX['moral']
    within = lambda idx: float(np.mean([norm[i, j] for i in idx for j in idx if i != j]))
    across = lambda A, Bx: float(np.mean([norm[i, j] for i in A for j in Bx]))
    # All THREE cross-category pairings, not just textual-vs-moral. The textual-moral
    # pair is the most separated one the methodology table can produce, so reporting it
    # alone as "the across-category figure" overstates how cleanly the three categories
    # come apart. Textual-vs-structural in fact runs HIGHER than within-moral, which is
    # why the bench's co-invocation structure is two blocs (textual+structural against
    # moral) rather than three categories -- the same split as the live/dead line.
    rows.append(dict(seed=s, within_textual=within(T), within_structural=within(S), within_moral=within(M),
                     across_textual_moral=across(T, M), across_textual_structural=across(T, S),
                     across_structural_moral=across(S, M)))
df5 = pd.DataFrame(rows); df5.to_csv(f'{OUTDIR}/exp5_coinvocation.csv', index=False)
summary['exp5'] = {c: float(df5[c].mean()) for c in df5.columns if c != 'seed'}
_wi = float(np.mean([df5[c].mean() for c in ['within_textual', 'within_structural', 'within_moral']]))
_ac = float(np.mean([df5[c].mean() for c in ['across_textual_moral', 'across_textual_structural', 'across_structural_moral']]))
summary['exp5']['within_mean'] = _wi; summary['exp5']['across_mean'] = _ac
summary['exp5']['within_across_ratio'] = _wi / _ac
print({c: round(df5[c].mean(), 3) for c in df5.columns if c != 'seed'})
print(f'within mean {_wi:.3f} vs across mean {_ac:.3f} -> ratio {_wi/_ac:.2f}')
print('NB: textual-vs-structural co-invocation exceeds within-moral, so the structure is')
print('    two blocs (textual+structural / moral), not three separated categories.')

{'within_textual': np.float64(0.798), 'within_structural': np.float64(0.635), 'within_moral': np.float64(0.551), 'across_textual_moral': np.float64(0.212), 'across_textual_structural': np.float64(0.664), 'across_structural_moral': np.float64(0.378)}
within mean 0.661 vs across mean 0.418 -> ratio 1.58
NB: textual-vs-structural co-invocation exceeds within-moral, so the structure is
    two blocs (textual+structural / moral), not three separated categories.


*Reading.* Within-category co-invocation exceeds across-category, consistent with
Notebook 0's finding that real cases carry correlated activations. This is a
consistency check, not an emergence result, since the co-invocation pattern is
partly built into the methodology table.

## Experiment 6 - Hartigan's dip test on the uptake shares

Experiment 1 measured bimodality with a two-versus-one Gaussian-mixture BIC
comparison (`gmm_bimodal`). That is a *model-based* test: it asks whether two
Gaussians fit better than one. Hartigan's dip test is *distribution-free*: it
measures the largest departure of the empirical CDF from the closest unimodal
CDF, so it makes no assumption that the humps are Gaussian.

Running both matters because the manuscript rests a hedge on the per-run result,
and the two tests disagree sharply on how sensitive they are at nine points.


In [10]:
try:
    import diptest
except ImportError:                      # Colab
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'diptest'], check=True)
    import diptest

RAW = [f'raw_{p}' for p, _ in NAMED_PRINCIPLES]

# (a) Per-run: nine shares per run, which is the claim the manuscript makes.
per = []
for _, r in df1.iterrows():
    x = np.sort(r[RAW].values.astype(float))
    dip, pval = diptest.diptest(x)
    per.append({'regime': r['regime'], 'seed': int(r['seed']), 'dip': float(dip), 'p': float(pval)})
dfd = pd.DataFrame(per)
dfd.to_csv(f'{OUTDIR}/exp6_diptest.csv', index=False)

rej_all = int((dfd.p < 0.05).sum())
rej_static = int((dfd[dfd.regime == 'static'].p < 0.05).sum())
n_static = int((dfd.regime == 'static').sum())

# (b) Pooled: every share from every run, where the test has power.
pool = df1[RAW].values.ravel().astype(float)
dip_pool, p_pool = diptest.diptest(pool)
pool_static = df1[df1.regime == 'static'][RAW].values.ravel().astype(float)
dip_st, p_st = diptest.diptest(pool_static)

summary['exp6_diptest'] = {
    'per_run_reject_of_total': [rej_all, int(len(dfd))],
    'per_run_reject_static': [rej_static, n_static],
    'per_run_p_median': float(dfd.p.median()),
    'pooled_all': {'n': int(pool.size), 'dip': float(dip_pool), 'p': float(p_pool)},
    'pooled_static': {'n': int(pool_static.size), 'dip': float(dip_st), 'p': float(p_st)},
    'gmm_per_run_reject': int(summary['exp1']['per_seed_bimodal_of_120']),
}

print(f"per-run dip test (n=9): rejects unimodality on {rej_all} of {len(dfd)} runs at p<0.05")
print(f"  static bench alone: {rej_static} of {n_static} | median p {dfd.p.median():.3f}")
print(f"pooled, all runs   (n={pool.size}): dip={dip_pool:.5f}  p={p_pool:.2e}")
print(f"pooled, static only (n={pool_static.size}): dip={dip_st:.5f}  p={p_st:.2e}")
print(f"-> the GMM/BIC measure rejects on {summary['exp1']['per_seed_bimodal_of_120']} of 120 per run;")
print("   the dip test rejects far more often, and both reject decisively when pooled.")

assert p_pool < 0.01, 'pooled dip test must reject unimodality'
assert p_st < 0.01, 'static-bench pooled dip test must reject unimodality'


per-run dip test (n=9): rejects unimodality on 66 of 120 runs at p<0.05
  static bench alone: 11 of 30 | median p 0.047
pooled, all runs   (n=1080): dip=0.07352  p=0.00e+00
pooled, static only (n=270): dip=0.12822  p=0.00e+00
-> the GMM/BIC measure rejects on 3 of 120 per run;
   the dip test rejects far more often, and both reject decisively when pooled.


*Reading.* The two tests agree where it counts and differ where the sample is
too small to carry either. Pooled, both reject unimodality decisively. Per run,
on nine points, the dip test rejects on about half the runs while the
Gaussian-mixture criterion rejects on three. Neither per-run figure should carry
the two-group claim; the pooled result does.


In [11]:
json.dump(summary, open(f'{OUTDIR}/notebook1_summary.json', 'w'), indent=2)
print(json.dumps(summary, indent=2)[:1500])

{
  "exp1": {
    "emerged": true,
    "membership_stable": true,
    "membership_same_across_rules": true,
    "always_live": [
      "text",
      "original",
      "precedent",
      "structure",
      "federalism"
    ],
    "always_dead": [
      "dignity",
      "equality",
      "liberty",
      "welfare"
    ],
    "mean_cluster_gap": 0.5120638231278163,
    "gap_by_regime": {
      "appointment": 0.47193280457973896,
      "fitness": 0.6633832045323679,
      "noise": 0.4521992702099925,
      "static": 0.46074001318916574
    },
    "per_seed_bimodal_of_120": 3,
    "pooled_gmm_dbic": 664.1441318064775,
    "pooled_gmm_bimodal": true,
    "pooled_bc": 0.6570587693560486,
    "p_static_vs_appointment": 0.0426517054438591,
    "p_static_vs_noise": 0.17719271034002304,
    "p_static_vs_fitness": 1.862645149230957e-09
  },
  "exp2": {
    "gap_by_spread": {
      "medium": 0.2571192035821737,
      "narrow": 0.13526265838588417,
      "wide": 0.36068255387727094
    },
    "nlive

## Self-check
Re-derives the soundness properties and compares headline numbers to a committed
reference. Raises an error on drift.

In [12]:
ok = True
def check(name, cond):
    global ok; ok = ok and bool(cond); print(('PASS' if cond else 'FAIL'), '-', name)

# model code is imported from the module, not pasted into the notebook
check('core is imported, not copied', is_determinate.__module__ == 'berman_core' and LegalSystem.__module__ == 'berman_core')
# the ABM is reproducible
_a = LegalSystem(turnover='fitness', seed=4); _a.simulate(800)
_b = LegalSystem(turnover='fitness', seed=4); _b.simulate(800)
check('ABM reproducible at fixed seed', np.array_equal(_a.system_weights, _b.system_weights))
# emergence really happened (not the uniform init), and was measured after simulate()
check('weights emerged (moved off uniform 0.5)', summary['exp1']['emerged'])
check('live/dead membership identical across seeds and rules', summary['exp1']['membership_same_across_rules'])
check('separation present under STATIC (zero selection)', summary['exp1']['gap_by_regime']['static'] > 0.2)
check('pooled distribution is bimodal (GMM)', summary['exp1']['pooled_gmm_bimodal'])
# the nonlexicality metric is not leaking (pure lexical rule scores exactly 0)
_w = named_weight_vector('moderate', 0); _A = random_activations(2000, 9, np.random.default_rng(0))
def _topmag(X):
    X = np.atleast_2d(X); j = np.argmax(np.abs(X), axis=1); return np.sign(X[np.arange(len(X)), j]).astype(int)
check('nonlexicality metric: pure lexical rule scores exactly 0', nonlexicality_score(_topmag, _A, seed=1) == 0.0)
# emergent weights pass R3 and R4 under static
check('emergent weights pass R3 (static, all seeds)', summary['exp3']['static']['R3'] == 1.0)
check('emergent weights pass R4 (static, mean margin > 0)', summary['exp3']['static']['mean_margin'] > 0)
check('single determinacy threshold', DEFAULT_TAU == 0.20)
# headline values within tolerance of the committed reference
REF = dict(gap_static=REF_GAP_STATIC, gap_fitness=REF_GAP_FITNESS, pooled_dbic=REF_POOLED_DBIC,
           sigma_static=REF_SIGMA_STATIC, margin_static=REF_MARGIN_STATIC)
check('exp1 static gap ~ ref', abs(summary['exp1']['gap_by_regime']['static'] - REF['gap_static']) < 0.03)
check('exp1 fitness gap ~ ref', abs(summary['exp1']['gap_by_regime']['fitness'] - REF['gap_fitness']) < 0.03)
check('exp3 static mean sigma ~ ref', abs(summary['exp3']['static']['mean_sigma'] - REF['sigma_static']) < 0.02)
print('\n' + ('ALL CHECKS PASSED' if ok else 'SOME CHECKS FAILED'))
assert ok, 'self-check failed'

PASS - core is imported, not copied
PASS - ABM reproducible at fixed seed
PASS - weights emerged (moved off uniform 0.5)
PASS - live/dead membership identical across seeds and rules
PASS - separation present under STATIC (zero selection)
PASS - pooled distribution is bimodal (GMM)
PASS - nonlexicality metric: pure lexical rule scores exactly 0
PASS - emergent weights pass R3 (static, all seeds)
PASS - emergent weights pass R4 (static, mean margin > 0)
PASS - single determinacy threshold
PASS - exp1 static gap ~ ref
PASS - exp1 fitness gap ~ ref
PASS - exp3 static mean sigma ~ ref

ALL CHECKS PASSED


## What this notebook establishes

Given a heterogeneous bench, deciding cases produces a stable separation of the
principles into a high-uptake group and a low-uptake group, with the same
membership every time. Applying Berman's uptake threshold turns this into the
legal/sub-legal boundary his account requires. The separation does not depend on
how judges are replaced over time, including not replacing them at all, so it is
a property of the bench's heterogeneity, not of any optimizer. The emergent
weights satisfy the same requirements the hand-set weights did in Notebook 0.

What this does not establish is that practice grounds the weights in Berman's
full sense. The emergent quantity is uptake share, invocation frequency off a
fixed set of methodologies. Berman's weight is conferred by a principle winning
conflicts and by being relied upon in later decisions; a principle invoked often
but always overruled would have high uptake share and low weight. That the
replacement rule does not matter also means the result is driven by the
stipulated methodology mix, not by anything judges learn. Both points mark the
work for Notebook 1c, which implements weight conferred by reliance and by
winning conflicts, and for Notebook 2, which compares this mechanism against
strategic learning.